In [2]:

from pathlib import Path

def validate_dataset(label_dir):
    label_dir = Path(label_dir)

    bad_files = []
    total_files = 0
    total_boxes = 0
    bad_boxes = 0
    empty_files = 0

    for file in label_dir.rglob("*.txt"):
        total_files += 1
        has_valid_box = False

        with open(file, "r") as f:
            lines = f.readlines()

        if len(lines) == 0:
            empty_files += 1
            continue

        for i, line in enumerate(lines):
            parts = line.strip().split()

            # ❌ wrong format
            if len(parts) != 5:
                bad_boxes += 1
                bad_files.append((file, f"line {i}: wrong format -> {line.strip()}"))
                continue

            try:
                c, x, y, w, h = map(float, parts)
            except:
                bad_boxes += 1
                bad_files.append((file, f"line {i}: non-float values -> {line.strip()}"))
                continue

            total_boxes += 1

            # ❌ invalid geometry
            if w <= 0 or h <= 0:
                bad_boxes += 1
                bad_files.append((file, f"line {i}: zero/negative size -> {line.strip()}"))
                continue

            # ❌ out of bounds YOLO format
            if not (0 <= x <= 1 and 0 <= y <= 1):
                bad_boxes += 1
                bad_files.append((file, f"line {i}: coords out of range -> {line.strip()}"))
                continue

            has_valid_box = True

        if not has_valid_box:
            bad_files.append((file, "no valid boxes in file"))

    # ===== REPORT =====
    print("\n===== DATASET VALIDATION REPORT =====")
    print(f"Total label files: {total_files}")
    print(f"Total boxes: {total_boxes}")
    print(f"Empty files: {empty_files}")
    print(f"Bad boxes: {bad_boxes}")
    print(f"Bad files: {len(set([f for f, _ in bad_files]))}")

    print("\n===== SAMPLE ERRORS =====")
    for f, msg in bad_files[:20]:
        print(f"{f} -> {msg}")

In [3]:
validate_dataset("../data/train/labels")


===== DATASET VALIDATION REPORT =====
Total label files: 7691
Total boxes: 9019
Empty files: 0
Bad boxes: 1
Bad files: 1

===== SAMPLE ERRORS =====
..\data\train\labels\train_01480.txt -> line 0: coords out of range -> 0 2.872587 1.338918 0.108108 0.185567


In [13]:
with open("../data/train/labels/train_01480.txt", "r") as f:
    content = f.read()
    print(content)

0 0.486486 0.559923 0.694981 0.568299
